import sys, os
from pathlib import Path

# Resolve project root regardless of where Jupyter was launched from
_root = Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))

import anthropic
from sentence_transformers import SentenceTransformer

from ingestor import load_locomo
from memory_writer import MemoryWriter
from memory_store import MemoryStore
from context_builder import ContextBuilder, RetrievalConfig
from evaluator import answer_question

# Initialize
client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Initialized.')

In [ ]:
# Load dataset
conversations = load_locomo(_root / 'data' / 'locomo10.json')
conv = conversations[0]
print(f'Working with session: {conv.session_id}')
print(f'Turns: {len(conv.turns)}, QA pairs: {len(conv.qa_pairs)}')

In [ ]:
# Load dataset
conversations = load_locomo('../data/locomo10.json')
conv = conversations[0]
print(f'Working with session: {conv.session_id}')
print(f'Turns: {len(conv.turns)}, QA pairs: {len(conv.qa_pairs)}')

In [ ]:
# ── Step 1: Extract memories ──────────────────
writer = MemoryWriter(
    client=client,
    embedder=embedder,
    window_size=4,
    stride=2,
    importance_threshold=0.3,
    dedup_threshold=0.92,
)

print('Extracting memories (this calls OpenAI API)...')
memories = writer.process_conversation(conv, verbose=True)
print(f'\nExtracted {len(memories)} memories')

In [ ]:
# Inspect extracted memories
print('Top 10 memories by importance:\n')
for m in sorted(memories, key=lambda x: x.importance, reverse=True)[:10]:
    print(f'  [{m.type:12s}] importance={m.importance:.2f}  persistence={m.persistence}')
    print(f'  "{m.fact}"')
    print(f'  source_turns={m.source_turns}')
    print()

In [ ]:
# ── Step 2: Store in Chroma ───────────────────
store = MemoryStore(collection_name='demo', embedder=embedder)
store.add_batch(memories)
print(f'Stored {store.count()} memories in Chroma')

In [ ]:
# ── Step 3: Retrieve for a query ─────────────
config = RetrievalConfig(
    alpha=0.5, beta=0.3, gamma=0.1, lam=0.1,
    token_budget=500,
)
builder = ContextBuilder(store=store, config=config)

# Use first QA pair as example
qa = conv.qa_pairs[0]
print(f'Question: {qa.question}')
print(f'Gold answer: {qa.answer}')
print(f'Evidence turns: {qa.evidence_turn_ids}')
print()

context, debug = builder.build_context(
    query=qa.question,
    session_id=conv.session_id,
    verbose=True,
)
print()
print('=== Retrieved Context ===')
print(context)

In [ ]:
# Scoring breakdown
print('Scoring breakdown for retrieved memories:\n')
for d in debug:
    print(f'  score={d["score"]:.3f}  rel={d["relevance"]:.3f}  '
          f'imp={d["importance"]:.2f}  rec={d["recency"]:.2f}')
    print(f'  "{d["fact"][:80]}"')
    print()

In [ ]:
# ── Step 4: Answer the question ───────────────
predicted = answer_question(client, qa.question, context)
print(f'Question:  {qa.question}')
print(f'Gold:      {qa.answer}')
print(f'Predicted: {predicted}')

In [ ]:
# ── Step 5: Score it ─────────────────────────
from evaluator import exact_match, token_f1

em = exact_match(predicted, qa.answer)
f1 = token_f1(predicted, qa.answer)
print(f'Exact match: {em}')
print(f'Token F1:    {f1:.3f}')

In [ ]:
# ── Interactive: try your own query ──────────
my_query = 'What decisions were made about the project?'  # change this

context, _ = builder.build_context(query=my_query, session_id=conv.session_id)
answer = answer_question(client, my_query, context)
print(f'Q: {my_query}')
print(f'A: {answer}')
print()
print('Context used:')
print(context)